In [10]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math

In [11]:
FILE = 'output.csv'

"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)


In [12]:
cols_dropped = ['sugg.select.utime', 'sugg.response.utime', 'dec.location.category']

sdf = df.drop(columns=cols_dropped)

print(sdf.columns)

cols = ['uid', 'decision_idx', 'datetime', 'date', 'day_slot', 'is_randomized', 'avail', 'send', 'send_active', 'send_sedentary', 'returned_message', 'response', 'activity', 'location', 'weather', 'temperature', 'jbsteps30', 'jbsteps30pre']



sdf.columns = cols

print(sdf.columns)

Index(['user.index', 'decision.index.nogap', 'datetime', 'date',
       'sugg.select.slot', 'is.randomized', 'avail', 'send', 'send.active',
       'send.sedentary', 'returned.message', 'response', 'recognized.activity',
       'location_group', 'dec.weather.condition', 'dec.temperature',
       'jbsteps30', 'jbsteps30pre'],
      dtype='str')
Index(['uid', 'decision_idx', 'datetime', 'date', 'day_slot', 'is_randomized',
       'avail', 'send', 'send_active', 'send_sedentary', 'returned_message',
       'response', 'activity', 'location', 'weather', 'temperature',
       'jbsteps30', 'jbsteps30pre'],
      dtype='str')


In [13]:
sdf.isnull().sum()
sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

uid  datetime           
2    2015-08-02 22:00:00    1
3    2015-08-06 20:30:00    1
6    2015-08-17 16:00:00    1
7    2015-08-21 22:00:00    1
     2015-08-21 23:30:00    1
                           ..
33   2016-01-01 16:00:00    1
     2016-01-13 14:05:00    1
35   2015-12-15 12:00:00    1
37   2015-12-15 12:00:00    1
     2016-01-13 22:25:00    1
Length: 105, dtype: int64

In [14]:


# df.fillna()
# df.median()
# df.interpolate()

sdf['datetime'] = pd.to_datetime(sdf['datetime'])
sdf = sdf.set_index('datetime')

sdf['temperature'] = sdf.groupby('uid')['temperature'].transform(
    lambda x: x.interpolate(method='time').ffill().bfill()
)

# 3. 恢复索引
sdf = sdf.reset_index()

sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

Series([], dtype: int64)

In [15]:
# 先把 unknown 和报错字符串替换为 NaN
sdf['weather'] = sdf['weather'].replace(
    ['unknown',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.getJSONObject(JSONObject.java:516)',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.<init>(JSONObject.java:179)'],
    pd.NA
)


sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()

# 用同一用户同一天的众数填补
def fill_mode(x):
    mode = x.mode()
    return x.fillna(mode[0] if len(mode) > 0 else pd.NA)

sdf['weather'] = sdf.groupby(['uid', 'date'])['weather'].transform(fill_mode)


# 剩余的用前向填充
sdf['weather'] = sdf.groupby('uid')['weather'].ffill()

sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()



sdf.isnull().sum()

datetime               0
uid                    0
decision_idx           0
date                   0
day_slot               0
is_randomized          0
avail                  0
send                   0
send_active            0
send_sedentary         0
returned_message       0
response            4017
activity               0
location               0
weather                0
temperature            0
jbsteps30              0
jbsteps30pre           0
dtype: int64

In [16]:
'''
send:
 - 0: no send
 - 1: active
 - 2: sedentary

when send.active == False and send.sedentary == False, returned.message is always 'donotnotify'
when send == False, returned.message is always 'donotnotify'
when returned.message == 'donotnotify', send == False
'''
# sdf[sdf['send'] == 0][['send_active', 'send_sedentary']].value_counts()
# sdf[sdf['send_active'] == 0 & (sdf['send_sedentary'] == 0)]['send'].value_counts()
sdf['send'] = sdf['send_active'].astype(int) + sdf['send_sedentary'].astype(int) * 2
sdf['send'].value_counts()

sdf.drop(columns=['send_active', 'send_sedentary'], inplace=True)

sdf.columns

Index(['datetime', 'uid', 'decision_idx', 'date', 'day_slot', 'is_randomized',
       'avail', 'send', 'returned_message', 'response', 'activity', 'location',
       'weather', 'temperature', 'jbsteps30', 'jbsteps30pre'],
      dtype='str')

In [17]:
sdf[sdf['response'].isnull()]['send'].value_counts()
# sdf[sdf['response'].isnull()]['returned_message'].value_counts()


sdf.loc[sdf['response'].isnull() & (sdf['send'] == 0), 'response'] = 'no_send'
sdf[sdf['response'].isnull()]['send'].value_counts()

sdf.loc[sdf['response'].isnull() & (sdf['send'] != 0), 'response'] = 'no_response'

In [26]:
# print(sdf.isnull().sum())

sdf['date'] = pd.to_datetime(sdf['date'])

sdf['study_day'] = (sdf['date'] - sdf.groupby('uid')['date'].transform('min')).dt.days + 1

print(sdf)

sdf.to_csv('./cleaned_output.csv', index=False)

                datetime  uid  decision_idx       date  day_slot  \
0    2015-07-22 16:30:00    1           0.0 2015-07-22         2   
1    2015-07-22 18:30:00    1           1.0 2015-07-22         3   
2    2015-07-22 21:30:00    1           2.0 2015-07-22         4   
3    2015-07-22 23:30:00    1           3.0 2015-07-22         5   
4    2015-07-23 09:30:00    1           4.0 2015-07-23         1   
...                  ...  ...           ...        ...       ...   
6723 2016-01-25 22:25:00   37         224.0 2016-01-25         4   
6724 2016-01-26 00:02:00   37         225.0 2016-01-26         5   
6725 2016-01-26 13:00:00   37         226.0 2016-01-26         1   
6726 2016-01-26 16:26:00   37         227.0 2016-01-26         2   
6727 2016-01-26 19:00:00   37         228.0 2016-01-26         3   

      is_randomized  avail  send  \
0              True   True     2   
1              True   True     1   
2             False   True     0   
3              True   True     2   
4  